### **Setup: Imports und Initialisierung**

In [2]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from quinex_application.quinex import Quinex
from quinex_application.visualisation_metadata import VisualisationMetadata
from quinex_application.mongos_store import MongoStore

In [3]:
quinex = Quinex()
visualisation = VisualisationMetadata("../data/paper/paper_dataset.parquet")
mongodb = MongoStore(quinex, database_name="quinex_db_query4", collection_name="papers_2020")

### **Create Bulk Analysis**

In [4]:
name = "quinex_application_query4"
response = quinex.intialize_analysis(name)
print(response)

{'detail': 'Analysis with name quinex_application_query4 already exists. Please choose another name.'}


### **Import and Sort Paper by Year**

In [5]:
visualisation.load_df( query='Query: TITLE-ABS-KEY ( ( "energy system*" ) AND ( simul* OR model* OR "plan" OR "plans" OR "planning" OR optimi* OR analy* OR assess* OR evaluat* ) ) AND ( LIMIT-TO ( OA , "all" ) ) AND ( LIMIT-TO ( LANGUAGE , "English" ) )', date="14.05.2025")
df_doi = visualisation.df

In [6]:
# Sort by year
dfs_by_year = {
    year: group_df
    for year, group_df in df_doi.groupby("Year")
}
df_2020 = dfs_by_year[2020]
print(len(df_2020))

2286


### **Add Papers via DOI**

In [14]:
mongodb.insert_paper(name, df_2020)

Verarbeitet: 1624/2286 | ✅ Neu: 925 | 🔁 Vorhanden: 0 | ❌ Fehlgeschlagen: 699

ConnectionError: ('Connection aborted.', ConnectionAbortedError(10053, 'An established connection was aborted by the software in your host machine', None, 10053, None))

#### **Finish remaining Papers**

In [7]:
# Hole die DOIs (_id) aus beiden Collections als Sets
papers_2020_ids = {doc["_id"] for doc in mongodb.get_collection("papers_2020")}
failed_papers_ids = {doc["_id"] for doc in mongodb.get_collection("failed_papers")}

# Vereine beide Sets (alle DOIs, die ausgeschlossen werden sollen)
excluded_ids = papers_2020_ids.union(failed_papers_ids)

# Filtere das DataFrame: nur Zeilen mit DOIs, die NICHT in excluded_ids enthalten sind
filtered_df = df_2020[~df_2020["DOI"].isin(excluded_ids)]
print(len(filtered_df))

457


In [ ]:
mongodb.insert_paper(name, filtered_df)

Verarbeitet: 164/457 | ✅ Neu: 101 | 🔁 Vorhanden: 0 | ❌ Fehlgeschlagen: 63

### **Successfully Added Papers**

In [ ]:
dois_df = mongodb.get_papers_from_collection(df_doi, "papers_2020")
print(len(dois_df))

### **Perform Bulk Analysis**

In [ ]:
# Vor Start:
# 9283806    normal quinex_b j-goepfe  R 15-02:07:14      1 cn31
# 9083722    normal quinex_b j-goepfe  R 15-02:06:41      1 cn26

# Nach Start:
# 9283806    normal quinex_b j-goepfe  R 15-03:19:29      1 cn31
# 9083722    normal quinex_b j-goepfe  R 15-03:18:56      1 cn26
# 11064122    normal quinex_b j-goepfe  R       1:19      1 cn26
# 11064123    normal quinex_b j-goepfe  R       1:19      1 cn26
# 11064124    normal quinex_b j-goepfe  R       1:19      1 cn30
# 11064125    normal quinex_b j-goepfe  R       1:19      1 cn30
# 11064126    normal quinex_b j-goepfe  R       1:19      1 cn30
# 11064127    normal quinex_b j-goepfe  R       1:19      1 cn30
# 11064128    normal quinex_b j-goepfe  R       1:19      1 cn30
# 11064129    normal quinex_b j-goepfe  R       1:19      1 cn30

In [ ]:
dois = [doi for doi in dois_df["DOI"]]
mongodb.reset_quinex_result(dois)
mongodb.process_paper(name, dois, number_of_gpus=8)

### **Add Bulk Analysis to MongoDB**

In [ ]:
dois = [doi for doi in dois_df["DOI"]]
mongodb.add_analysis(name, dois)

In [ ]:
analysis = mongodb.get_collection("papers_2021")

### **Update Paper Parquet**

In [ ]:
df = mongodb.update_paper_parquet("papers_2021", "../data/paper/paper_dataset.parquet", "18.06.2025", "17.04.2025")

In [ ]:
display(df[(df["Quinex"] == True)])